# Customer Churn SQL Analysis & Data Validation

## Project Overview
This notebook uses SQL and SQLite to analyze the customer churn dataset, validate data quality, and extract business insights. The customer data is organized into two tables: `customers` and `services`, connected through the `customerID` field.

## Objectives
- Connect to the existing SQLite database, `customer_churn.db`.
- Inspect database tables and validate record counts.
- Check customer ID uniqueness and customer-service relationships.
- Join customer and service information using SQL JOIN operations.
- Calculate customer churn rates by contract type using GROUP BY and aggregate functions.
- Use Common Table Expressions (CTEs) to filter contract groups with higher churn rates.
- Apply window functions to rank contract types by churn rate.
- Interpret the results to identify customer segments associated with higher churn.

## Tools & Technologies
- SQL
- SQLite
- Python
- Pandas
- Jupyter Notebook

## Database
The database contains two tables:

- **customers:** Customer demographics, tenure, contract, billing details, and churn status.
- **services:** Phone, internet, security, backup, support, and streaming service information.

Both tables contain 7,043 records. Validation confirms 7,043 unique customer IDs and zero customer records without a matching service record.

## Key Business Insight
The SQL analysis found that month-to-month contracts have a substantially higher observed churn rate than longer-term contracts. This relationship can help identify customer segments for further investigation, but it does not prove that contract type alone causes churn.

## Expected Outcome
This notebook demonstrates SQL querying, data validation, table joins, aggregation, CTEs, and window functions while generating reliable insights to support customer churn analysis.

**Note:** Run the notebook using the existing project database path. Do not recreate or overwrite the database unless necessary.


In [1]:
import sqlite3
import pandas as pd
import os

db_path = os.path.join("outputs", "customer_churn.db")

conn = sqlite3.connect(db_path)

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

print("Tables in database:")
display(tables)

Tables in database:


,name
0,customers
1,services


In [2]:
import sqlite3
import pandas as pd
import os

db_path = os.path.join("outputs", "customer_churn.db")
conn = sqlite3.connect(db_path)

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
""", conn)

display(tables)

,name
0,customers
1,services


In [3]:
# Check the structure and number of rows in each table

for table_name in ["customers", "services"]:
    print(f"\n{'=' * 40}")
    print(f"TABLE: {table_name}")
    print("=" * 40)

    # Display column names and data types
    schema = pd.read_sql_query(
        f"PRAGMA table_info({table_name})",
        conn
    )
    display(schema[["name", "type"]])

    # Count rows
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total_rows FROM {table_name}",
        conn
    )
    print("Total rows:", count.iloc[0]["total_rows"])

    # Display first 5 rows
    sample = pd.read_sql_query(
        f"SELECT * FROM {table_name} LIMIT 5",
        conn
    )
    display(sample)


TABLE: customers


,name,type
0,customerID,TEXT
1,gender,TEXT
2,SeniorCitizen,INTEGER
3,Partner,TEXT
4,Dependents,TEXT
5,tenure,INTEGER
6,Contract,TEXT
7,PaperlessBilling,TEXT
8,PaymentMethod,TEXT
9,MonthlyCharges,REAL


Total rows: 7043


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



TABLE: services


,name,type
0,customerID,TEXT
1,PhoneService,TEXT
2,MultipleLines,TEXT
3,InternetService,TEXT
4,OnlineSecurity,TEXT
5,OnlineBackup,TEXT
6,DeviceProtection,TEXT
7,TechSupport,TEXT
8,StreamingTV,TEXT
9,StreamingMovies,TEXT


Total rows: 7043


,customerID,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,No,No phone service,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,Yes,No,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,Yes,No,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,No,No phone service,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Yes,No,Fiber optic,No,No,No,No,No,No


In [4]:
query = """
SELECT
    c.customerID,
    c.gender,
    c.tenure,
    c.Contract,
    c.MonthlyCharges,
    c.Churn,
    s.InternetService,
    s.TechSupport,
    s.OnlineSecurity
FROM customers AS c
INNER JOIN services AS s
    ON c.customerID = s.customerID
LIMIT 10;
"""

joined_data = pd.read_sql_query(query, conn)

display(joined_data)

,customerID,gender,tenure,Contract,MonthlyCharges,Churn,InternetService,TechSupport,OnlineSecurity
0,7590-VHVEG,Female,1,Month-to-month,29.85,No,DSL,No,No
1,5575-GNVDE,Male,34,One year,56.95,No,DSL,No,Yes
2,3668-QPYBK,Male,2,Month-to-month,53.85,Yes,DSL,No,Yes
3,7795-CFOCW,Male,45,One year,42.30,No,DSL,Yes,Yes
4,9237-HQITU,Female,2,Month-to-month,70.70,Yes,Fiber optic,No,No
5,9305-CDSKC,Female,8,Month-to-month,99.65,Yes,Fiber optic,No,No
6,1452-KIOVK,Male,22,Month-to-month,89.10,No,Fiber optic,No,No
7,6713-OKOMC,Female,10,Month-to-month,29.75,No,DSL,No,Yes
8,7892-POOKP,Female,28,Month-to-month,104.80,Yes,Fiber optic,Yes,No
9,6388-TABGU,Male,62,One year,56.15,No,DSL,No,Yes


In [5]:
query = """
SELECT
    Contract,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Percent
FROM customers
GROUP BY Contract
ORDER BY Churn_Rate_Percent DESC;
"""

contract_churn = pd.read_sql_query(query, conn)

display(contract_churn)

,Contract,Total_Customers,Churned_Customers,Churn_Rate_Percent
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [6]:
query = """
WITH ContractSummary AS (
    SELECT
        Contract,
        COUNT(*) AS Total_Customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
            AS Churned_Customers
    FROM customers
    GROUP BY Contract
),
ChurnAnalysis AS (
    SELECT
        Contract,
        Total_Customers,
        Churned_Customers,
        ROUND(
            100.0 * Churned_Customers / Total_Customers,
            2
        ) AS Churn_Rate_Percent
    FROM ContractSummary
)
SELECT *
FROM ChurnAnalysis
WHERE Churn_Rate_Percent > 10
ORDER BY Churn_Rate_Percent DESC;
"""

cte_result = pd.read_sql_query(query, conn)
display(cte_result)

,Contract,Total_Customers,Churned_Customers,Churn_Rate_Percent
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27


In [7]:
query = """
WITH ContractSummary AS (
    SELECT
        Contract,
        COUNT(*) AS Total_Customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
            AS Churned_Customers
    FROM customers
    GROUP BY Contract
),
ChurnRates AS (
    SELECT
        Contract,
        Total_Customers,
        Churned_Customers,
        ROUND(
            100.0 * Churned_Customers / Total_Customers,
            2
        ) AS Churn_Rate_Percent
    FROM ContractSummary
)
SELECT
    Contract,
    Total_Customers,
    Churned_Customers,
    Churn_Rate_Percent,
    RANK() OVER (
        ORDER BY Churn_Rate_Percent DESC
    ) AS Churn_Rank
FROM ChurnRates
ORDER BY Churn_Rank;
"""

window_result = pd.read_sql_query(query, conn)
display(window_result)

,Contract,Total_Customers,Churned_Customers,Churn_Rate_Percent,Churn_Rank
0,Month-to-month,3875,1655,42.71,1
1,One year,1473,166,11.27,2
2,Two year,1695,48,2.83,3


In [8]:
validation_query = """
SELECT
    (SELECT COUNT(*) FROM customers) AS Customer_Rows,
    (SELECT COUNT(DISTINCT customerID) FROM customers)
        AS Unique_Customers,
    (SELECT COUNT(*) FROM services) AS Service_Rows,
    (
        SELECT COUNT(*)
        FROM customers c
        LEFT JOIN services s
            ON c.customerID = s.customerID
        WHERE s.customerID IS NULL
    ) AS Customers_Without_Services;
"""

validation = pd.read_sql_query(validation_query, conn)
display(validation)

,Customer_Rows,Unique_Customers,Service_Rows,Customers_Without_Services
0,7043,7043,7043,0


In [9]:
print("Customers table:")
display(pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM customers",
    conn
))

print("Services table:")
display(pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM services",
    conn
))

print("First 5 customers:")
display(pd.read_sql_query(
    "SELECT * FROM customers LIMIT 5",
    conn
))

Customers table:


,total_rows
0,7043


Services table:


,total_rows
0,7043


First 5 customers:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
validation_query = """
SELECT
    (SELECT COUNT(*) FROM customers) AS Customer_Rows,
    (SELECT COUNT(DISTINCT customerID) FROM customers)
        AS Unique_Customers,
    (SELECT COUNT(*) FROM services) AS Service_Rows,
    (
        SELECT COUNT(*)
        FROM customers c
        LEFT JOIN services s
            ON c.customerID = s.customerID
        WHERE s.customerID IS NULL
    ) AS Customers_Without_Services;
"""

validation = pd.read_sql_query(validation_query, conn)
display(validation)

,Customer_Rows,Unique_Customers,Service_Rows,Customers_Without_Services
0,7043,7043,7043,0


In [11]:
print(validation.to_string(index=False))

 Customer_Rows  Unique_Customers  Service_Rows  Customers_Without_Services
          7043              7043          7043                           0
